In [ ]:
import jax.numpy as jnp
import jax
import numpy as np
import time
import gymnasium as gym
import exciting_environments as excenv
#from exciting_environments import GymWrapper, EnvironmentRegistry
#import jax_dataclasses as jdc
from exciting_environments import EnvironmentRegistry
from dataclasses import fields
from exciting_environments.utils import MinMaxNormalization
jax.config.update("jax_enable_x64", True)
import diffrax
from exciting_environments.pmsm.motor_parameters import MotorVariant

In [ ]:
pend_env=EnvironmentRegistry.PENDULUM.make()

## Step and Simulate ahead

In [ ]:
key=jax.random.PRNGKey(1234)
obs, state = pend_env.reset()

In [ ]:
obs,states,last_state=pend_env.sim_ahead(state,jnp.ones((4,1)))
obs

In [ ]:
pend_env.generate_rew_trunc_term_ahead(states,jnp.ones((4,1)))

In [ ]:
key=jax.random.PRNGKey(1234)
obs, state = pend_env.reset()
generated_observations = []
generated_actions= []
generated_observations.append(obs)
for i in range(4):
    key,subkey= jax.random.split(key)
    action = jnp.ones(2)#jax.random.uniform(subkey,(2,),minval=-1,maxval=1)
    obs, state = pend_env.step(state, action)
    generated_actions.append(action)
    generated_observations.append(obs)

In [ ]:
generated_observations

### Vmapped

In [ ]:
batch_size= 2
keys = jax.random.split(jax.random.PRNGKey(0), batch_size)
envs = [EnvironmentRegistry.PENDULUM.make(static_params={"g": jnp.array(9.81), "l": jnp.array(1.0),  "m": jnp.array(1.0)}),EnvironmentRegistry.PENDULUM.make(static_params={"g": jnp.array(9.81), "l": jnp.array(1.0),  "m": jnp.array(1.0)})]
batched_envs = jax.tree.map(lambda *args: jnp.stack(args), *envs)
#batched_envs = EnvironmentRegistry.PENDULUM.make(batch_size=2)
single_env = EnvironmentRegistry.PENDULUM.make()

In [ ]:
keys = jax.random.split(jax.random.PRNGKey(0), batch_size)
obs, states = batched_envs.vmap_reset(keys)
print(obs)

In [ ]:
state_in=batched_envs.vmap_generate_state_from_observation(obs,keys)
state_in

In [ ]:
actions = jnp.ones((batch_size,5))
obs, states = batched_envs.vmap_reset()
next_obs, next_states = batched_envs.vmap_step(states, actions)
print(next_obs)

In [ ]:
keys = jax.random.split(jax.random.PRNGKey(0), batch_size)
obs, states = batched_envs.vmap_reset()
actions = jnp.ones((batch_size,4,1))
next_obs, next_states, last_state = batched_envs.vmap_sim_ahead(states, actions)
print(next_obs)

In [ ]:
#print(jax.vmap(lambda e, s, a: e.generate_rew_trunc_term_ahead(s, a))(batched_envs, next_states, actions))
batched_envs.vmap_generate_rew_trunc_term_ahead(next_states, actions)